<a href="https://colab.research.google.com/github/Jun-1112/FYP-project-Trunk-and-weed-detection-for-agricultural-usage/blob/main/Segmentation(mini_model%2B_final_model).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics

from google.colab import drive
import os

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Notebooks/predicted_annotations_full.zip'
extract_dir = '/content/cvat_extracted'

!unzip -q "{zip_path}" -d "{extract_dir}"
print("Extracted contents:", os.listdir(extract_dir))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 1.6 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

In [ ]:
import xml.etree.ElementTree as ET
import cv2
import yaml

video_path = '/content/drive/MyDrive/Colab Notebooks/GX011313.MP4'
xml_path = '/content/cvat_extracted/annotations.xml'

output_img_dir = '/content/dataset/images/train'
output_lbl_dir = '/content/dataset/labels/train'
os.makedirs(output_img_dir, exist_ok=True)
os.makedirs(output_lbl_dir, exist_ok=True)

tree = ET.parse(xml_path)
root = tree.getroot()

class_map = {'Palm Tree Trunk': 0}  # weed class dropped, trunk-only mini-model
print("Classes:", class_map)

frame_annotations = {}  # raw_frame_number -> list of (class_id, points)

for track in root.findall('.//track'):
    label_name = track.get('label')
    if label_name not in class_map:
        continue
    class_id = class_map[label_name]

    for polygon in track.findall('polygon'):
        if polygon.get('keyframe') != '1':
            continue  # skip interpolated (non-manually-verified) frames
        if polygon.get('outside') == '1':
            continue  # object marked as not present on this frame

        frame_num = int(polygon.get('frame'))
        points_str = polygon.get('points')

        frame_annotations.setdefault(frame_num, [])
        pts = []
        for pair in points_str.split(';'):
            x, y = map(float, pair.split(','))
            pts.extend([x, y])
        frame_annotations[frame_num].append((class_id, pts))

print("Seed frames found:", sorted(frame_annotations.keys()))
print("Seed frame count:", len(frame_annotations))

cap = cv2.VideoCapture(video_path)
orig_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

extracted = 0
for frame_num in sorted(frame_annotations.keys()):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if not ret:
        print(f"Could not read frame {frame_num}")
        continue

    cv2.imwrite(f"{output_img_dir}/frame_{frame_num:06d}.jpg", frame)

    lines = []
    for class_id, pts in frame_annotations[frame_num]:
        norm = [f"{pts[i] / orig_width:.6f} {pts[i+1] / orig_height:.6f}" for i in range(0, len(pts), 2)]
        lines.append(f"{class_id} " + " ".join(norm))

    with open(f"{output_lbl_dir}/frame_{frame_num:06d}.txt", 'w') as f:
        f.write("\n".join(lines))
    extracted += 1

cap.release()
print(f"Extracted {extracted} seed frames with matching labels.")

data_yaml = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/train',  # throwaway prototype — no held-out split needed
    'names': {i: name for name, i in class_map.items()}
}
with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
print("Created /content/dataset/data.yaml")

In [1]:
#mini-model training
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=60,
    imgsz=640,
    batch=8,
    patience=15,
    device=0,
)

In [ ]:
# To determine which task frames still need auto-annotation
STEP = 15  # matches the actual CVAT task sampling interval

labeled_task_indices = {f // STEP for f in frame_annotations.keys()}
print("Already labeled manually task indices:", sorted(labeled_task_indices))

all_task_indices = list(range(0, 831))  # CVAT task spans 831 frames: 0 - 830
to_predict_indices = [i for i in all_task_indices if i not in labeled_task_indices]
print("Frames to auto-annotate:", len(to_predict_indices))


In [ ]:
# To auto-annotate remaining frames using the mini-model
import cv2

dst_root = '/content/cvat_import_ultralytics_full'
img_dir = f'{dst_root}/images/Train'
lbl_dir = f'{dst_root}/labels/Train'
os.makedirs(img_dir, exist_ok=True)
os.makedirs(lbl_dir, exist_ok=True)

model = YOLO('/content/runs/segment/train/weights/best.pt')  # the mini-model
cap = cv2.VideoCapture(video_path)

def read_frame_robust(cap, video_path, target_frame, window=10):
    """Handles occasional frame-seek failures by falling back to a nearby
    frame within a small window and reading sequentially from there."""
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()
    if ret:
        return frame
    cap2 = cv2.VideoCapture(video_path)
    start = max(0, target_frame - window)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, start)
    frame = None
    for i in range(start, target_frame + 1):
        ret, f = cap2.read()
        if not ret:
            break
        frame = f
    cap2.release()
    return frame

failed = []
for task_idx in to_predict_indices:
    raw_frame = task_idx * STEP
    frame = read_frame_robust(cap, video_path, raw_frame)
    if frame is None:
        failed.append(task_idx)
        continue

    img_path = f'{img_dir}/frame_{task_idx:06d}.jpg'
    cv2.imwrite(img_path, frame)

    r = model.predict(source=img_path, conf=0.25, imgsz=640, device=0, verbose=False)[0]
    lines = []
    if r.masks is not None:
        for cls_id, poly in zip(r.boxes.cls.tolist(), r.masks.xyn):
            coords = ' '.join(f'{x:.6f} {y:.6f}' for x, y in poly)
            lines.append(f'{int(cls_id)} {coords}')

    with open(f'{lbl_dir}/frame_{task_idx:06d}.txt', 'w') as f:
        f.write('\n'.join(lines))

cap.release()
print(f"Predicted {len(to_predict_indices) - len(failed)} frames")
print("Failed frames (task indices):", failed)

In [ ]:
# To reimported back the refined full annotations zips
import glob, random, shutil

refined_zip = '/content/drive/MyDrive/Colab Notebooks/predicted_annotations_full.zip'
extract_dir = '/content/cvat_extracted'

!unzip -q "{refined_zip}" -d "{extract_dir}"
print("Extracted refined dataset:", os.listdir(extract_dir))

In [ ]:
# To build train/val split for the refined full annotation dataset
src_img_dir = '/content/cvat_extracted/images/Train'
src_lbl_dir = '/content/cvat_extracted/labels/Train'

out_root = '/content/full_dataset'
for split in ['train', 'val']:
    os.makedirs(f'{out_root}/images/{split}', exist_ok=True)
    os.makedirs(f'{out_root}/labels/{split}', exist_ok=True)

img_files = sorted(glob.glob(f'{src_img_dir}/*.jpg'))
print("Total images found:", len(img_files))

random.seed(0)
random.shuffle(img_files)
val_count = max(1, int(len(img_files) * 0.15))
val_set = set(img_files[:val_count])

for img_path in img_files:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = f'{src_lbl_dir}/{stem}.txt'
    split = 'val' if img_path in val_set else 'train'
    shutil.copy(img_path, f'{out_root}/images/{split}/{stem}.jpg')
    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, f'{out_root}/labels/{split}/{stem}.txt')
    else:
        open(f'{out_root}/labels/{split}/{stem}.txt', 'w').close()

print(f"Train: {len(img_files) - val_count}, Val: {val_count}")

data_yaml = {
    'path': '/content/full_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'Palm Tree Trunk'}
}
with open('/content/full_dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
print(open('/content/full_dataset/data.yaml').read())

In [ ]:
# To train the final trunk segmentation model (Cycle 2)
model = YOLO('yolov8n-seg.pt')
results = model.train(
    data='/content/full_dataset/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    patience=25,
    device=0,
    plots=True,
    project='/content/runs/segment',
    name='seg_model_for_treetrunk',
)

In [ ]:
# To back up the final model and training artifacts to Drive
backup_dir = '/content/drive/MyDrive/Colab Notebooks/fyp_backup'
os.makedirs(backup_dir, exist_ok=True)

run_dir = '/content/runs/segment/seg_model_for_treetrunk'

shutil.copy(f'{run_dir}/weights/best.pt', f'{backup_dir}/bestseg.pt')
shutil.copy(f'{run_dir}/weights/last.pt', f'{backup_dir}/lastseg.pt')
shutil.copy('/content/full_dataset/data.yaml', f'{backup_dir}/data.yaml')
shutil.copy(f'{run_dir}/results.csv', f'{backup_dir}/results.csv')
shutil.copy(f'{run_dir}/args.yaml', f'{backup_dir}/args.yaml')

for f in os.listdir(run_dir):
    if f.endswith('.png') or f.endswith('.jpg'):
        shutil.copy(f'{run_dir}/{f}', f'{backup_dir}/{f}')

print("Model weights, data.yaml, results.csv, args.yaml, and plots backed up to Drive.")